# Primi modelli: un albero per prezzi simulati

Materiale originale **AI Lab**, non una lezione Kaggle. Codice in inglese; attività e spiegazioni in italiano.

Esegui le celle in ordine con **Maiusc + Invio**. Salva una copia prima di modificare il codice.


## Obiettivi e prerequisiti
Conoscere liste, funzioni e cicli. Distinguere feature, target e previsione. I dati immobiliari sono **simulati**: non sono Melbourne o Iowa. Unità monetarie arbitrarie. Nessun download o pacchetto esterno.


In [ ]:
import random
from statistics import mean

rng = random.Random(42)
rows = []
for _ in range(240):
    area = rng.uniform(30, 180)
    rooms = rng.randint(1, 6)
    price = 40000 + 1800 * area + 10000 * rooms + rng.gauss(0, 16000)
    rows.append(([area, rooms], price))
rng.shuffle(rows)
train_rows = rows[:144]
valid_rows = rows[144:192]
test_rows = rows[192:]
assert len(train_rows) + len(valid_rows) + len(test_rows) == len(rows)
print('Training / validation / test:', len(train_rows), len(valid_rows), len(test_rows))


## Un albero piccolo
La funzione cerca divisioni che riducono la somma degli errori quadratici. Una foglia restituisce il prezzo medio. Questo è codice originale scolastico, semplificato rispetto a scikit-learn.


In [ ]:
def fit_tree(data, depth=0, max_depth=3):
    targets = [target for _, target in data]
    if depth >= max_depth or len(data) < 8:
        return mean(targets)
    best = None
    for feature in range(len(data[0][0])):
        values = sorted(set(features[feature] for features, _ in data))
        step = max(1, len(values) // 16)
        for threshold in values[1::step]:
            left = [row for row in data if row[0][feature] <= threshold]
            right = [row for row in data if row[0][feature] > threshold]
            if not left or not right:
                continue
            left_mean = mean(target for _, target in left)
            right_mean = mean(target for _, target in right)
            loss = sum((target - left_mean) ** 2 for _, target in left)
            loss += sum((target - right_mean) ** 2 for _, target in right)
            if best is None or loss < best[0]:
                best = (loss, feature, threshold, left, right)
    if best is None:
        return mean(targets)
    _, feature, threshold, left, right = best
    return (feature, threshold, fit_tree(left, depth+1, max_depth),
            fit_tree(right, depth+1, max_depth))


def predict_tree(model, features):
    if isinstance(model, (float, int)):
        return float(model)
    feature, threshold, left, right = model
    return predict_tree(left if features[feature] <= threshold else right, features)


def mae(model, data):
    return mean(abs(target - predict_tree(model, features)) for features, target in data)


In [ ]:
model = fit_tree(train_rows, max_depth=3)
print("Validation MAE:", mae(model, valid_rows))
print("Prediction for area=90, rooms=3:", predict_tree(model, [90, 3]))
assert mae(model, valid_rows) < mae(mean(target for _, target in train_rows), valid_rows)


## Esercizi
1. Descrivi feature e target.
2. Cambia la casa da prevedere.
3. Prova profondità 1 e 5; annota il MAE di validazione.

## Riepilogo
Un modello apprende dai dati di training e produce previsioni per nuovi esempi. La qualità deve essere confrontata con una baseline.
